## Points to improve
- ~smoothen slope using 5th and 95th percentiles~
- ~convert decibels to linear~
- ~test code in Kashmir (valley)~
- push low level code to utils and other relevant modules
- final checks on documentation

## Generated mean and sd for the following in this script:
- 44R
- 46R
- 47R
- 44Q
- 44P
- 42R
- 43R

## East India zones
'44P', '44Q', '44R', '45Q', '45R', '46Q', '46R', '47R'

In [1]:
import sys
sys.path.append('../')

from ifmiap import flood_mapper
from ifmiap import utils
import geopandas as gpd
import time

In [2]:
# small piece of code to get zone-wise list of IDs
gdf = gpd.read_file(r'../resources/india_utm_fishnet_buffer.gpkg')
zone_id_group = gdf[['zone', 'ID']].groupby('zone')['ID'].apply(list)

zone_id_dict = dict()

for idx in zone_id_group.index:
    zone_id_dict[idx] = zone_id_group[idx]

In [ ]:
%%time

#for wet_period in ['2017/07', '2021/07', '2022/07', '2023/07']:
for year in [2017, 2018, 2019, 2020, 2021, 2022, 2023]:
    for month in ['08', '09', '10']:
        wet_period = f'{year}/{month}'
        
        for n, zone_id in enumerate(zone_id_dict.keys()):
            print(f'Processing: {zone_id}. {n} out of {len(zone_id_dict)}.')
            t1 = time.time()
            
            # create the flood mapper class
            flood_mapper_obj = flood_mapper(
                grid_shapefile = r'../resources/india_utm_fishnet_buffer.gpkg',
                grid_id_list = zone_id_dict[zone_id],#[ID]
                dry_date_col = 'dry_month',
                id_col = 'ID',
                dry_years=[2018, 2022],
                slope_dir = r'../resources/slope/',
                wet_duration = [wet_period, wet_period]
            )
            
            flood_mapper_obj.get_dry_dates()
            
            if len(flood_mapper_obj.aoi_ids_to_process) > 0:
                flood_mapper_obj.generate_dry_date_ranges()
                flood_mapper_obj.get_s1_items(dry_wet='dry')
                flood_mapper_obj.read_scenes(dry_wet='dry', overview_level=2)
                flood_mapper_obj.generate_mean_std_by_aoi()
            else:
                flood_mapper_obj.load_mean_std_by_aoi()
            
            flood_mapper_obj.prepare_slope(dem_overview=0, buffer=500)
            
            flood_mapper_obj.prepare_wet_scenes(overview_level=2)
            flood_mapper_obj.generate_number_of_scenes(export_raster=True)
            flood_mapper_obj.map_floods(vv_thd=-2.5, vh_thd=-2.5, rel_slope_thd=20,
                                          export_raster=False, export_vector=True, export_maps=False)
            flood_mapper_obj.merge_floods_by_date(export_raster=True)
            flood_mapper_obj.monthly_sum()
            
            t2 = time.time()
            t_delta = t2 - t1
            print(f'Time taken for {zone_id} zone: {(t_delta / 60):.2f} mins.\n')

Processing: 42Q. 0 out of 15.
Following previously processed AOI IDs found. Will be skipped. If you are running with a new year range, consider modifying or deleting the json file.
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Previously processed ../output/mean_std/2018_2022_aoi_1_vv_vh_mean_std.nc read successfully!
Previously processed ../output/mean_std/2018_2022_aoi_2_vv_vh_mean_std.nc read successfully!
Previously processed ../output/mean_std/2018_2022_aoi_3_vv_vh_mean_std.nc read successfully!
Previously processed ../output/mean_std/2018_2022_aoi_4_vv_vh_mean_std.nc read successfully!
Previously processed ../output/mean_std/2018_2022_aoi_5_vv_vh_mean_std.nc read successfully!
Previously processed ../output/mean_std/2018_2022_aoi_6_vv_vh_mean_std.nc read successfully!
Previously processed ../output/mean_std/2018_2022_aoi_7_vv_vh_mean_std.nc read successfully!
Previously processed ../output/mean_std/2018_2022_aoi_8_vv_vh_mean_std.nc read successfully!
Previously processed ../output

/usr/local/lib/python3.8/dist-packages/pystac_client/item_search.py:835: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


In [6]:
import shutil, glob

shutil.make_archive('../', 'zip', r'../ifmiap/')

'/home/pratyusht/datadrive/ifmiap/scripts.zip'

In [16]:
import shutil, glob

shutil.make_archive('../output/flood_monthlyadded_201907', 'zip', r'/home/pratyusht/datadrive/ifmiap/output/flood_raster/monthlyadded/',
                   *glob.glob(r'/home/pratyusht/datadrive/ifmiap/output/output/flood_raster/monthlyadded/*WET_201907_201907*')[:2])

'/home/pratyusht/datadrive/ifmiap/output/flood_monthlyadded_201907.zip'

In [ ]:
#!sudo shutdown now